## SalesForce Pro Inc - Revenue Intelligence System

In [2]:
# data cleaning

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# load raw file — change path if your folder structure is different
df = pd.read_csv('salesforce_pro_raw.csv')

# first thing i always do — just get a feel for the shape
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print(df.dtypes)
print(df.head(3))

Rows: 50,500 | Columns: 22
order_id            object
order_date          object
ship_date           object
delivery_date       object
customer_id         object
customer_name       object
segment             object
region              object
state               object
sales_rep_id        object
category            object
sub_category        object
product_name        object
quantity             int64
unit_price         float64
discount           float64
sales_amount       float64
profit             float64
payment_method      object
ship_mode           object
return_flag          int64
customer_rating    float64
dtype: object
    order_id  order_date   ship_date delivery_date customer_id  \
0  ORD017864  2022-02-19  2022-02-22    2022-02-25   CUST02439   
1  ORD009550  2024-11-26  2024-12-09    2024-12-11   CUST06324   
2  ORD045511  2024-07-10  2024-07-10    2024-07-13   CUST07230   

    customer_name     segment region           state sales_rep_id  ...  \
0     Vinay Gupta  Governm

In [5]:
df.isnull().sum()

order_id             0
order_date           0
ship_date            0
delivery_date        0
customer_id          0
customer_name       80
segment              0
region              60
state                0
sales_rep_id        72
category             0
sub_category         0
product_name         0
quantity             0
unit_price           0
discount            40
sales_amount         0
profit               0
payment_method      49
ship_mode            0
return_flag          0
customer_rating    100
dtype: int64

In [6]:
# filling the customer_name with 'Unknown' since it's a string 
df['customer_name'] = df['customer_name'].fillna('Unknown')

# filling the region with mode value since it's categorical 
df['region'] = df['region'].fillna(df['region'].mode()[0])

# filling the sales_rep_id with 'UNASSIGNED' since it's a string
df['sales_rep_id'] = df['sales_rep_id'].fillna('UNASSIGNED')

# filling the payment_method with mode value since it's categorical
df['payment_method'] = df['payment_method'].fillna(df['payment_method'].mode()[0])

# filling the customer_rating with median value since it's numerical
df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())

# discount missing — assuming no discount was applied
df['discount'] = df['discount'].fillna(0)


In [7]:
df.isnull().sum()

order_id           0
order_date         0
ship_date          0
delivery_date      0
customer_id        0
customer_name      0
segment            0
region             0
state              0
sales_rep_id       0
category           0
sub_category       0
product_name       0
quantity           0
unit_price         0
discount           0
sales_amount       0
profit             0
payment_method     0
ship_mode          0
return_flag        0
customer_rating    0
dtype: int64

In [9]:
# checking for duplicates

df.duplicated().sum()

492

In [10]:
df.drop_duplicates(inplace=True)

In [12]:
print(df['category'].unique())
print(df['category'].value_counts())

['Furniture' 'Software' 'Electronics' 'Clothing' 'Office Supplies'
 'CLOTHING' 'SOFTWARE' 'OFFICE SUPPLIES' 'ELECTRONICS' 'FURNITURE']
category
Office Supplies    10096
Software           10005
Furniture           9996
Electronics         9849
Clothing            9762
OFFICE SUPPLIES       76
ELECTRONICS           59
FURNITURE             58
SOFTWARE              56
CLOTHING              51
Name: count, dtype: int64


In [13]:
# Some category values are all caps
df['category'] = df['category'].str.title()

In [14]:
print(df['category'].unique())

['Furniture' 'Software' 'Electronics' 'Clothing' 'Office Supplies']


In [15]:
# Sales amount have some negative values which is not possible.

df = df[df['sales_amount'] > 0]

In [16]:
df.shape

(49858, 22)

In [17]:
# some discount values are greater than one like 20,30 which may be entered as percentage , a data entry issue.

df.loc[df['discount'] > 1, 'discount'] = df.loc[df['discount'] > 1, 'discount'] / 100

In [18]:
# converting date columns to datetime
df['order_date']    = pd.to_datetime(df['order_date'])
df['ship_date']     = pd.to_datetime(df['ship_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])

In [19]:
# does ship always come after order?
print((df['ship_date'] < df['order_date']).sum())

50


In [21]:
# ship_date before order_date is clearly wrong , dropping those rows
bad_dates = df[df['ship_date'] < df['order_date']]

df = df[df['ship_date'] >= df['order_date']]

In [22]:
# I noticed some quantity values were abnormally high , using IQR method to find and remove outliers

Q1 = df['quantity'].quantile(0.25)
Q3 = df['quantity'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print('lower bound:', lower)
print('upper bound:', upper)
print('outliers found:', ((df['quantity'] < lower) | (df['quantity'] > upper)).sum())

# removing the outliers
df = df[(df['quantity'] >= lower) & (df['quantity'] <= upper)]


lower bound: -7.5
upper bound: 28.5
outliers found: 80


In [23]:
df.shape

(49728, 22)

In [24]:
df['order_year']  = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_quarter'] = df['order_date'].dt.quarter


In [25]:
# delivery time in days 
df['delivery_days'] = (df['delivery_date'] - df['order_date']).dt.days

In [27]:
# profit margin % per order
df['profit_margin'] = (df['profit'] / df['sales_amount'] * 100).round(2)

Final Check

In [28]:
df.isnull().sum()

order_id           0
order_date         0
ship_date          0
delivery_date      0
customer_id        0
customer_name      0
segment            0
region             0
state              0
sales_rep_id       0
category           0
sub_category       0
product_name       0
quantity           0
unit_price         0
discount           0
sales_amount       0
profit             0
payment_method     0
ship_mode          0
return_flag        0
customer_rating    0
order_year         0
order_month        0
order_quarter      0
delivery_days      0
profit_margin      0
dtype: int64

In [29]:
df.shape

(49728, 27)

In [30]:
print('date range:')
print('from:', df['order_date'].min())
print('to:', df['order_date'].max())

date range:
from: 2021-01-01 00:00:00
to: 2024-12-31 00:00:00


In [31]:
print('categories:')
print(df['category'].value_counts())
print()

categories:
category
Office Supplies    10111
Software           10011
Furniture           9996
Electronics         9854
Clothing            9756
Name: count, dtype: int64



In [32]:
df.to_csv('salesforce_pro_cleaned.csv', index=False)